In [7]:
import pandas as pd
import ast
from tqdm.auto import tqdm
import re
from collections import Counter
import pickle
import os

import pymorphy3

morph = pymorphy3.MorphAnalyzer()

tqdm.pandas()

PROCESSED_TRAIN_PATH = "../../data/processed/train_bio.csv"
CATALOG_PATH = "../../data/external/products_5ka_raw.json"

OUTPUT_DIR = "../../data/external/"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "master_dictionary.pkl")

os.makedirs(OUTPUT_DIR, exist_ok=True)

Загрузка

In [8]:
df = pd.read_csv(PROCESSED_TRAIN_PATH, sep=";")
df["tokens"] = df["tokens"].progress_apply(ast.literal_eval)
df["tags"] = df["tags"].progress_apply(ast.literal_eval)
print("Данные успешно загружены и преобразованы.")

  0%|          | 0/23163 [00:00<?, ?it/s]

  0%|          | 0/23163 [00:00<?, ?it/s]

Данные успешно загружены и преобразованы.


Извлечение сущностей

In [9]:
def extract_entities_from_bio(tokens: list, tags: list, target_entity_type: str) -> set:
    entities = set()
    current_entity_tokens = []

    for token, tag in zip(tokens, tags):
        if tag.startswith(f"B-{target_entity_type}"):
            if current_entity_tokens:
                entities.add(" ".join(current_entity_tokens))
            current_entity_tokens = [token]
        elif tag.startswith(f"I-{target_entity_type}"):
            if current_entity_tokens:
                current_entity_tokens.append(token)
            else:
                current_entity_tokens = [token]
        else:
            if current_entity_tokens:
                entities.add(" ".join(current_entity_tokens))
                current_entity_tokens = []
    
    if current_entity_tokens:
        entities.add(" ".join(current_entity_tokens))
        
    return entities

In [10]:
brand_sets = df.progress_apply(
    lambda row: extract_entities_from_bio(row['tokens'], row['tags'], 'BRAND'),
    axis=1
)

all_brands_set = set()
for s in tqdm(brand_sets, desc="Объединение результатов"):
    all_brands_set.update(s)

print(f"\nНайдено {len(all_brands_set)} уникальных брендов.")

known_brands_from_train = [brand.strip() for brand in all_brands_set if brand.strip()]
known_brands_from_train.sort(key=len, reverse=True)

print("Информация: Список брендов успешно создан и отсортирован по длине.")
print("\nПример 10 самых длинных брендов:")
print(known_brands_from_train[:10])

  0%|          | 0/23163 [00:00<?, ?it/s]

Объединение результатов:   0%|          | 0/23163 [00:00<?, ?it/s]


Найдено 3748 уникальных брендов.
Информация: Список брендов успешно создан и отсортирован по длине.

Пример 10 самых длинных брендов:
['▁а . рост а гро ком п лек', '▁а . рост а гро комплекс', "▁ar khan gel ' sk kh leb", '▁а . рост а гро ком пле', "▁ar khan gel ' s kk hle", '▁б . ю . а лек сандр ов', '▁б . ю . а лек сандр о', "▁ar khan gel ' s kk hl", '▁вас иле ост ров ское', '▁vas ile os rov sko e']


Извлечение типов

In [11]:
type_sets_from_train = df.progress_apply(
    lambda row: extract_entities_from_bio(row['tokens'], row['tags'], 'TYPE'),
    axis=1
)

types_from_train_set = set()
for s in tqdm(type_sets_from_train, desc="Объединение результатов"):
    types_from_train_set.update(s)

print(f"\nНайдено {len(types_from_train_set)} уникальных типов в train.csv.")
print("\nПример 10 случайных типов из обучающей выборки:")
print(list(types_from_train_set)[:10])

  0%|          | 0/23163 [00:00<?, ?it/s]

Объединение результатов:   0%|          | 0/23163 [00:00<?, ?it/s]


Найдено 13648 уникальных типов в train.csv.

Пример 10 случайных типов из обучающей выборки:
['▁шоколад', '▁мар оны', '▁мас дин ы', '▁juice', '▁кур ице й', '▁ оборудован и', '▁газ ирован я', '▁картоф ел', '▁и ы ло', '▁го рох ч']


In [12]:
def atomize_category(category_name: str) -> set:
    atoms = re.split(r'[,\/]| и ', category_name)
    
    lemmatized_atoms = set()
    for atom in atoms:
        clean_atom = atom.strip().lower()
        if not clean_atom:
            continue
            
        words_in_atom = clean_atom.split()
        for word in words_in_atom:
            if len(word) > 2 and not any(char.isdigit() for char in word):
                normal_form = morph.parse(word)[0].normal_form
                lemmatized_atoms.add(normal_form)
                
    return lemmatized_atoms

In [13]:
print("Информация: Загрузка каталога товаров...")
df_catalog = pd.read_json(CATALOG_PATH)
print(f"Загружено {len(df_catalog)} товаров из каталога.")

print("\nИнформация: Атомизация и извлечение типов из категорий каталога...")
types_from_catalog_sets = df_catalog['category_name'].progress_apply(atomize_category)

types_from_catalog_set = set()
for s in tqdm(types_from_catalog_sets, desc="Объединение результатов"):
    types_from_catalog_set.update(s)

print(f"Найдено {len(types_from_catalog_set)} уникальных атомарных типов в каталоге.")
print("\nПример 10 случайных атомарных типов из каталога:")
print(list(types_from_catalog_set)[:10])

Информация: Загрузка каталога товаров...
Загружено 13903 товаров из каталога.

Информация: Атомизация и извлечение типов из категорий каталога...


  0%|          | 0/13903 [00:00<?, ?it/s]

Объединение результатов:   0%|          | 0/13903 [00:00<?, ?it/s]

Найдено 167 уникальных атомарных типов в каталоге.

Пример 10 случайных атомарных типов из каталога:
['паста', 'крупа', 'варение', 'пятёрочка', 'перекус', 'готовый', 'выпечка', 'блюдо', 'мёд', 'десерт']


In [14]:
atomic_types_from_train_set = set()
for type_phrase in tqdm(types_from_train_set, desc="Атомизация типов из train"):
    words = type_phrase.split()
    for word in words:
        clean_word = word.strip().lower()
        if len(clean_word) > 2 and not any(char.isdigit() for char in clean_word):
             atomic_types_from_train_set.add(morph.parse(clean_word)[0].normal_form)

atomic_types = types_from_catalog_set.union(atomic_types_from_train_set)

print(f"\nИтоговый размер словаря атомарных типов: {len(atomic_types)}.")
print("Информация: Словарь типов успешно создан.")
print("\nПример 20 случайных типов из итогового словаря:")
print(sorted(list(atomic_types))[100:120])

Атомизация типов из train:   0%|          | 0/13648 [00:00<?, ?it/s]


Итоговый размер словаря атомарных типов: 2988.
Информация: Словарь типов успешно создан.

Пример 20 случайных типов из итогового словаря:
['ачи', 'ачка', 'аша', 'бад', 'бай', 'бак', 'бан', 'бап', 'бар', 'барни', 'бас', 'бат', 'бач', 'бег', 'без', 'бек', 'бел', 'бела', 'белить', 'бель']


Сбор словаря дескрипторов

In [15]:
o_tokens_counter = Counter()

for index, row in tqdm(df.iterrows(), total=len(df), desc="Сбор O-токенов"):
    tokens = row['tokens']
    tags = row['tags']
    for token, tag in zip(tokens, tags):
        if tag == 'O':
            o_tokens_counter.update([token.lower()])

print(f"\nНайдено {len(o_tokens_counter)} уникальных O-токенов.")
print("Топ-30 самых частотных O-токенов:")
print(o_tokens_counter.most_common(30))

Сбор O-токенов:   0%|          | 0/23163 [00:00<?, ?it/s]


Найдено 1782 уникальных O-токенов.
Топ-30 самых частотных O-токенов:
[('▁для', 667), ('▁с', 396), ('▁', 284), ('▁в', 183), ('а', 132), ('я', 128), ('▁без', 100), ('и', 73), ('▁до', 71), ('▁из', 70), ('й', 61), ('е', 60), ('▁со', 55), ('▁на', 54), ('ы', 54), ('ом', 53), ('▁по', 51), ('ки', 44), ('▁сахар', 43), ('▁посуд', 41), ('▁ко', 36), ('▁мы', 36), ('о', 35), ('▁к', 34), ('▁от', 33), ('-', 33), ('▁у', 32), ('ть', 31), ('ю', 31), ('▁мя', 30)]


In [16]:
TARGET_POS = {'ADJF', 'ADVB', 'PREP', 'CONJ', 'PRCL'}

def get_pos(tag):
    # Унифицированное извлечение части речи для pymorphy3 (и обратно совместимо, если потребуется)
    pos = getattr(tag, 'part_of_speech', None)
    if pos:
        return pos
    pos = getattr(tag, 'POS', None)
    if pos:
        return pos
    grammemes = getattr(tag, 'grammemes', None)
    if grammemes:
        # Возвращаем первый граммем, совпадающий с нашими целевыми POS
        for g in grammemes:
            if g in TARGET_POS:
                return g
    return None

stop_words_list = []
candidates = o_tokens_counter.most_common(300)

for word, count in tqdm(candidates, desc="Анализ частей речи"):
    if len(word) < 2:
        continue
    
    parsed_word = morph.parse(word)[0]
    pos = get_pos(parsed_word.tag)
    
    if pos in TARGET_POS:
        stop_words_list.append(word)

manual_additions = ['купить', 'цена', 'доставка', 'найти', 'заказать']
for word in manual_additions:
    if word not in stop_words_list:
        stop_words_list.append(word)
        
stop_words_set = set(stop_words_list)

print(f"\nСформирован словарь из {len(stop_words_set)} дескрипторов и стоп-слов.")
print("\nПример 20 слов из словаря:")
print(sorted(list(stop_words_set))[:20])

Анализ частей речи:   0%|          | 0/300 [00:00<?, ?it/s]


Сформирован словарь из 34 дескрипторов и стоп-слов.

Пример 20 слов из словаря:
['да', 'де', 'доставка', 'за', 'заказать', 'ка', 'ко', 'кой', 'купить', 'ле', 'ли', 'ль', 'на', 'найти', 'не', 'ни', 'но', 'ок', 'от', 'ской']


Финализация главного словаря

In [17]:
master_dictionary = {
    "known_brands": known_brands_from_train,
    "atomic_types": atomic_types,
    "stop_words": stop_words_set
}

print("\n--- Содержимое 'Главного Словаря' ---")
print(f"  Количество известных брендов: {len(master_dictionary['known_brands'])}")
print(f"  Количество атомарных типов: {len(master_dictionary['atomic_types'])}")
print(f"  Количество стоп-слов: {len(master_dictionary['stop_words'])}")
print("------------------------------------")

with open(OUTPUT_PATH, 'wb') as f:
    pickle.dump(master_dictionary, f)


--- Содержимое 'Главного Словаря' ---
  Количество известных брендов: 3748
  Количество атомарных типов: 2988
  Количество стоп-слов: 34
------------------------------------


Проверка

In [18]:
try:
    with open(OUTPUT_PATH, 'rb') as f:
        loaded_dictionary = pickle.load(f)
    
    assert "known_brands" in loaded_dictionary
    assert "atomic_types" in loaded_dictionary
    assert "stop_words" in loaded_dictionary
    assert isinstance(loaded_dictionary["known_brands"], list)
    assert isinstance(loaded_dictionary["atomic_types"], set)
    assert isinstance(loaded_dictionary["stop_words"], set)

    print("Проверка успешно пройдена! Файл корректен и содержит все необходимые данные.")
    
    print("\nПример брендов из загруженного словаря:")
    print(loaded_dictionary["known_brands"][:5])

except Exception as e:
    print(f"Ошибка при проверке файла: {e}")

✅ Проверка успешно пройдена! Файл корректен и содержит все необходимые данные.

Пример брендов из загруженного словаря:
['▁а . рост а гро ком п лек', '▁а . рост а гро комплекс', "▁ar khan gel ' sk kh leb", '▁а . рост а гро ком пле', "▁ar khan gel ' s kk hle"]
